# Project 10: Open-Domain Question Answering - From Lexical Retrieval to RAG

Group: 28

Dataset: rag-mini-wikipedia

## Setup
Install the retrieval libraries.

In [2]:
!pip -q install rank_bm25 sentence-transformers faiss-cpu datasets

## 1. Data
Load both configs of the dataset: the `text-corpus` (3,200 passages that every retriever searches) and the `question-answer` pairs (918 questions, used to evaluate every task). About 45% of the gold answers are just "yes" or "no".

In [3]:
# Step 1: load dataset and inspect
from datasets import load_dataset

corpus = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus")      # passages to search
qa     = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer")  # questions to evaluate
print(corpus); print(qa)

passages  = list(corpus[list(corpus.keys())[0]]["passage"])
questions = list(qa[list(qa.keys())[0]]["question"])
answers   = list(qa[list(qa.keys())[0]]["answer"])
print(len(passages), "passages |", len(questions), "questions")
print("PASSAGE 0:", passages[0][:300])
print("Q0:", questions[0], "->", answers[0])

# share of yes/no answers
yes_no = sum(a.strip().lower().rstrip(".!") in ("yes", "no") for a in answers)
print(yes_no, "yes/no =", round(yes_no / len(answers), 3))

README.md:   0%|          | 0.00/719 [00:00<?, ?B/s]

data/passages.parquet/part.0.parquet: reconstructing file:   0%|          |  0.00B /  797kB            

data/passages.parquet/part.0.parquet: downloading bytes:           |  0.00B            

Generating passages split:   0%|          | 0/3200 [00:00<?, ? examples/s]

data/test.parquet/part.0.parquet: reconstructing file:   0%|          |  0.00B / 54.4kB            

data/test.parquet/part.0.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/918 [00:00<?, ? examples/s]

DatasetDict({
    passages: Dataset({
        features: ['passage', 'id'],
        num_rows: 3200
    })
})
DatasetDict({
    test: Dataset({
        features: ['question', 'answer', 'id'],
        num_rows: 918
    })
})
3200 passages | 918 questions
PASSAGE 0: Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in the capital Montevideo and its metropolitan area.
Q0: Was Abraham Lincoln the sixteenth President of the United States? -> yes
409 yes/no = 0.446


## 2. Evaluation metrics
Answers are compared after SQuAD normalisation: lowercase, and remove punctuation, articles and extra spaces.
- **Exact Match (EM):** 1 if the normalised prediction equals the normalised gold answer, otherwise 0.
- **F1:** word-overlap F1 between prediction and gold, which gives partial credit.

In [4]:
import re, string
from collections import Counter

# SQuAD-style normalisation: lowercase, drop punctuation, articles and extra spaces
def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def exact_match(pred, gold):
    return float(normalize_answer(pred) == normalize_answer(gold))

# token-level F1: partial credit for overlapping words
def f1_score(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    if not p or not g:
        return float(p == g)
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0:
        return 0.0
    precision, recall = common / len(p), common / len(g)
    return 2 * precision * recall / (precision + recall)

print(normalize_answer("The  Eiffel Tower!"))
for pred, gold in [("Yes.", "yes"), ("the Mediterranean Sea", "Mediterranean"),
                   ("16th President", "sixteenth President"), ("Paris", "London")]:
    print(f"{pred!r:25} vs {gold!r:22} EM={exact_match(pred, gold)}  F1={f1_score(pred, gold):.2f}")

eiffel tower
'Yes.'                    vs 'yes'                  EM=1.0  F1=1.00
'the Mediterranean Sea'   vs 'Mediterranean'        EM=0.0  F1=0.67
'16th President'          vs 'sixteenth President'  EM=0.0  F1=0.50
'Paris'                   vs 'London'               EM=0.0  F1=0.00


# Task 1: Classical lexical retrieve-and-extract
## 3. BM25 retrieval
Passages and questions are tokenised (lowercase, alphanumeric, English stopwords removed). BM25 scores each passage by question-word matches. Rare words weigh more (IDF), and k1 and b control term-frequency saturation and length normalisation.

In [5]:
import re
import numpy as np
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# lowercase, keep letters/digits, drop common stopwords
def tokenize(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in ENGLISH_STOP_WORDS]

# build the index once over all passages (spec: k1=1.5, b=0.75)
bm25 = BM25Okapi([tokenize(p) for p in passages], k1=1.5, b=0.75)

# return passage indices sorted by BM25 score, best first
def bm25_search(question, k=5):
    scores = bm25.get_scores(tokenize(question))
    return list(np.argsort(-scores)[:k]), scores

q = questions[0]
print("Q:", q, "| tokens:", tokenize(q))
top, scores = bm25_search(q)
for rank, i in enumerate(top, 1):
    print(f"{rank}. [{i}] score={scores[i]:.2f}  {passages[i][:120]}")

Q: Was Abraham Lincoln the sixteenth President of the United States? | tokens: ['abraham', 'lincoln', 'sixteenth', 'president', 'united', 'states']
1. [322] score=18.84  Photograph showing the March 4, 1861, inauguration of Abraham Lincoln in front of United States Capitol.
2. [278] score=18.64  Abraham Lincoln (February 12, 1809 â April 15, 1865) was the sixteenth President of the United States, serving from Ma
3. [387] score=13.77  Abraham Lincoln's birthday, February 12, was formerly a national holiday, now commemorated as Presidents' Day. However, 
4. [364] score=13.62  The assassination of Abraham Lincoln. From left to right: Henry Rathbone, Clara Harris, Mary Todd Lincoln, Abraham Linco
5. [374] score=12.99  Lincoln appointed the following Justices to the Supreme Court of the United States:


## 4. Retrieval recall@k
There are no gold-passage labels, so a retrieval counts as a hit if the normalised gold answer appears as whole words in any of the top-k passages. Yes/no answers can't be string-matched and are excluded, which leaves n = 508. We keep the top 100 passages per question for reuse in Task 3.

In [6]:
from tqdm.auto import tqdm

# rank every question once; keep top-100 (reused for recall@10 and RRF in Task 3)
bm25_ranked = [bm25_search(q, k=100)[0] for q in tqdm(questions)]

# hit = normalised gold answer appears as whole words inside the passage
def answer_in_passage(gold, passage):
    return f" {normalize_answer(gold)} " in f" {normalize_answer(passage)} "

def is_yes_no(a):
    return normalize_answer(a) in ("yes", "no")

# recall@k over span-answer questions only
def recall_at_k(ranked, ks=(1, 3, 5, 10)):
    span_ids = [i for i, a in enumerate(answers) if not is_yes_no(a)]
    result = {}
    for k in ks:
        hits = 0
        for i in span_ids:
            top_k = ranked[i][:k]
            if any(answer_in_passage(answers[i], passages[p]) for p in top_k):
                hits += 1
        result[f"R@{k}"] = round(hits / len(span_ids), 3)
    result["n"] = len(span_ids)
    return result

print("BM25:", recall_at_k(bm25_ranked))

  0%|          | 0/918 [00:00<?, ?it/s]

BM25: {'R@1': 0.478, 'R@3': 0.571, 'R@5': 0.602, 'R@10': 0.63, 'n': 508}


## 5. Sentence selection
From the top-1 BM25 passage, choose the sentence that shares the most question words, weighting each shared word by its BM25 IDF.

In [7]:
# split a passage into sentences at . ! ? followed by a space and a capital/digit
def split_sentences(passage):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", passage) if s.strip()]

# sentence sharing the most (rare) question words; rare words weighted by BM25's IDF
def best_sentence(question, passage):
    q_words = set(tokenize(question))
    best, best_score = passage, -1
    for sent in split_sentences(passage):
        shared = q_words & set(tokenize(sent))
        score = sum(bm25.idf.get(w, 0) for w in shared)
        if score > best_score:
            best, best_score = sent, score
    return best

i = next(i for i, a in enumerate(answers) if not is_yes_no(a))   # first non-yes/no question
q = questions[i]
top1 = passages[bm25_ranked[i][0]]
print("Q:", q, "| GOLD:", answers[i])
for s in split_sentences(top1):
    print("-", s[:100])
print("\nBEST:", best_sentence(q, top1))

Q: How many long was Lincoln's formal education? | GOLD: 18 months
- There were few people who strongly or directly influenced Lincolnâs moral and intellectual develop
- There was no teacher, mentor, church leader, community leader, or peer that Lincoln would credit in 
- Lacking a formal education, Lincolnâs personal philosophy was shaped by "an amazingly retentive me
- Guelzo pg.
- 20.
- Miller pg.
- 57-59  Lincolnâs reading and study of the Bible was an integral part of his intellectual roots.

BEST: Lacking a formal education, Lincolnâs personal philosophy was shaped by "an amazingly retentive memory and a passion for reading and learning." It was Lincolnâs reading, rather than his relationships, that were most influential in shaping his personal beliefs.


## 6. Span extraction and Task 1 evaluation
No generation is allowed in Task 1, so a short answer is cut from the chosen sentence:
- Yes/no questions are answered "yes", the majority class.
- Otherwise, remove the question's words and stopwords, then take the first remaining chunk that matches the expected answer type ("how many"/"when" → contains a digit, "who"/"where"/"which" → a capitalised word), capped at 5 words.

In [8]:
YES_NO_START = ("is", "was", "are", "were", "do", "does", "did", "can", "could", "has", "have", "had", "will", "would")

# guess the expected answer type from the question wording
def question_type(q):
    q = q.lower()
    if q.startswith("how many") or q.startswith("how much"): return "number"
    if q.startswith("when") or "what year" in q:             return "date"
    if q.startswith(("who", "where", "which")):               return "name"
    return "other"

# does this chunk of words look like the expected type?
def fits(qtype, chunk):
    if qtype in ("number", "date"): return any(any(c.isdigit() for c in w) for w in chunk)
    if qtype == "name":             return any(w[0].isupper() for w in chunk)
    return True

def extract_answer(question, sentence, max_words=5):
    if question.lower().split()[0] in YES_NO_START:     # yes/no question -> majority class
        return "yes"
    q_words = set(re.findall(r"[a-z0-9]+", question.lower()))
    words = re.findall(r"[\w'-]+", sentence)
    # chunks = runs of consecutive sentence words that are NOT in the question
    chunks, cur = [], []
    for w in words:
        if w.lower() in q_words or w.lower() in ENGLISH_STOP_WORDS:
            if cur: chunks.append(cur); cur = []
        else:
            cur.append(w)
    if cur: chunks.append(cur)
    if not chunks:
        return sentence
    good = [c for c in chunks if fits(question_type(question), c)] or chunks
    return " ".join(good[0][:max_words])                  # first matching chunk, capped

# Task 1 on all questions: BM25 top-1 -> best sentence -> short span
preds = []
for i, q in enumerate(tqdm(questions)):
    sent = best_sentence(q, passages[bm25_ranked[i][0]])
    preds.append(extract_answer(q, sent))

def evaluate(preds, ids):
    em = sum(exact_match(preds[i], answers[i]) for i in ids) / len(ids)
    f1 = sum(f1_score(preds[i], answers[i]) for i in ids) / len(ids)
    return round(em, 3), round(f1, 3)

all_ids  = list(range(len(questions)))
span_ids = [i for i in all_ids if not is_yes_no(answers[i])]
yn_ids   = [i for i in all_ids if is_yes_no(answers[i])]
print("Task 1 EM/F1  all:", evaluate(preds, all_ids), " span:", evaluate(preds, span_ids), " yes/no:", evaluate(preds, yn_ids))
for i in span_ids[:5]:
    print(f"Q: {questions[i]}\n   pred: {preds[i]!r}   gold: {answers[i]!r}")

  0%|          | 0/918 [00:00<?, ?it/s]

Task 1 EM/F1  all: (0.413, 0.459)  span: (0.098, 0.182)  yes/no: (0.802, 0.802)
Q: How many long was Lincoln's formal education?
   pred: 'Lacking'   gold: '18 months'
Q: When did Lincoln begin his political career?
   pred: '1832'   gold: '1832'
Q: What did The Legal Tender Act of 1862 establish?
   pred: 'established'   gold: 'the United States Note, the first paper currency in United States history'
Q: Who suggested Lincoln grow a beard?
   pred: '11-year-old Grace Bedell'   gold: '11-year-old Grace Bedell'
Q: When did the Gettysburg address argue that America was born?
   pred: 'November 19 1863'   gold: '1776'


# Task 2: Retrieval-Augmented Generation
## 7. Dense retrieval
Every passage is embedded into a 384-dimensional vector with `all-MiniLM-L6-v2`. Vectors are L2-normalised, so the inner product in a FAISS `IndexFlatIP` equals cosine similarity, giving exact nearest-neighbour search. This is built over the same passages as BM25, with no re-chunking.

In [10]:
from sentence_transformers import SentenceTransformer
import faiss

# small sentence encoder: text -> 384-dim vector capturing meaning
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# embed every passage once; normalising makes inner product = cosine similarity
passage_vecs = encoder.encode(passages, batch_size=64, normalize_embeddings=True,
                              show_progress_bar=True).astype("float32")
print(passage_vecs.shape)

# flat = exact brute-force search, IP = inner product
index = faiss.IndexFlatIP(passage_vecs.shape[1])
index.add(passage_vecs)

def dense_search(question, k=5):
    q_vec = encoder.encode([question], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(q_vec, k)
    return ids[0].tolist(), scores[0]

# same question as Step 5: compare with BM25's top-1
q = questions[i]
top, scores = dense_search(q)
print("Q:", q, "| GOLD:", answers[i])
for rank, (p, s) in enumerate(zip(top, scores), 1):
    print(f"{rank}. [{p}] cos={s:.3f} has_answer={answer_in_passage(answers[i], passages[p])}  {passages[p][:90]}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (5 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

(3200, 384)
Q: When did the Gettysburg address argue that America was born? | GOLD: 1776
1. [358] cos=0.615 has_answer=True  In his Gettysburg Address Lincoln redefined the American nation, arguing that it was born 
2. [355] cos=0.610 has_answer=False  Lincoln's powerful rhetoric defined the issues of the war for the nation, the world, and p
3. [2212] cos=0.498 has_answer=False  Quotation from Woodrow Wilson's History of the American People as reproduced in the film T
4. [2227] cos=0.484 has_answer=False  * His earliest memory, from age 3, was of hearing that Abraham Lincoln had been elected an
5. [341] cos=0.476 has_answer=False  On March 4 1865, Lincoln delivered his second inaugural address, his favorite of all his s


## 8. Dense vs. BM25 recall
Dense retrieval for all questions, scored with the same recall@k as BM25.

In [11]:
# embed all questions at once, then take the top-100 passages for each (reused in Task 3)
q_vecs = encoder.encode(questions, batch_size=64, normalize_embeddings=True,
                        show_progress_bar=True).astype("float32")
_, dense_ids = index.search(q_vecs, 100)
dense_ranked = dense_ids.tolist()

print("BM25 :", recall_at_k(bm25_ranked))
print("Dense:", recall_at_k(dense_ranked))

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

BM25 : {'R@1': 0.478, 'R@3': 0.571, 'R@5': 0.602, 'R@10': 0.63, 'n': 508}
Dense: {'R@1': 0.398, 'R@3': 0.528, 'R@5': 0.589, 'R@10': 0.632, 'n': 508}


## 9. Generator
Qwen2.5-1.5B-Instruct receives the top-5 passages, each truncated to 200 tokens, in a chat prompt that asks for the shortest possible answer. Decoding is greedy (temperature 0) with at most 64 new tokens. Only the first line of the output is kept.

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

GEN = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(GEN)
tok.padding_side = "left"                      # needed for batched generation (Step 10)
llm = AutoModelForCausalLM.from_pretrained(GEN, dtype=torch.float16).to("cuda").eval()

SYSTEM = ("Answer the question using only the passages. Reply with the shortest possible answer "
          "(a name, number, date or short phrase), or just yes/no for yes/no questions.")

# keep only the first 200 tokens of a passage (spec)
def truncate(p, n=200):
    return tok.decode(tok(p, add_special_tokens=False)["input_ids"][:n])

def build_prompt(question, passage_ids):
    ctx = "\n\n".join(f"[{j+1}] {truncate(passages[p])}" for j, p in enumerate(passage_ids[:5]))
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Passages:\n{ctx}\n\nQuestion: {question}\nAnswer:"}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

# greedy decoding (= temperature 0), max 64 new tokens; keep the first line of the reply
@torch.no_grad()
def generate(prompts):
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
    out = llm.generate(**enc, max_new_tokens=64, do_sample=False, pad_token_id=tok.eos_token_id)
    new_tokens = out[:, enc["input_ids"].shape[1]:]
    return [t.strip().split("\n")[0].rstrip(".") for t in tok.batch_decode(new_tokens, skip_special_tokens=True)]

prompt = build_prompt(questions[i], dense_ranked[i])
print(prompt[:700], "...\n")
print("PRED:", generate([prompt])[0], "| GOLD:", answers[i])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

<|im_start|>system
Answer the question using only the passages. Reply with the shortest possible answer (a name, number, date or short phrase), or just yes/no for yes/no questions.<|im_end|>
<|im_start|>user
Passages:
[1] In his Gettysburg Address Lincoln redefined the American nation, arguing that it was born not in 1789 but in 1776, "conceived in Liberty, and dedicated to the proposition that all men are created equal." He declared that the sacrifices of battle had rededicated the nation to the propositions of democracy and equality, "that this nation shall have a new birth of freedom â and that government of the people, by the people, for the people, shall not perish from the earth." By ...

PRED: 1776 | GOLD: 1776


## 10. Task 2 evaluation
Generate answers for all 918 questions in batches of 16 (left padding), and score them with the same EM/F1 as Task 1.

In [13]:
# generate answers for every question, 16 prompts per batch
def answer_all(ranked, batch_size=16):
    preds = []
    for start in tqdm(range(0, len(questions), batch_size)):
        batch = range(start, min(start + batch_size, len(questions)))
        prompts = [build_prompt(questions[j], ranked[j]) for j in batch]
        preds.extend(generate(prompts))
    return preds

dense_preds = answer_all(dense_ranked)
print("Task 2 EM/F1  all:", evaluate(dense_preds, all_ids), " span:", evaluate(dense_preds, span_ids),
      " yes/no:", evaluate(dense_preds, yn_ids))
for j in span_ids[:5]:
    print(f"Q: {questions[j]}\n   pred: {dense_preds[j]!r}   gold: {answers[j]!r}")

  0%|          | 0/58 [00:00<?, ?it/s]

Task 2 EM/F1  all: (0.55, 0.641)  span: (0.295, 0.46)  yes/no: (0.866, 0.866)
Q: How many long was Lincoln's formal education?
   pred: '18 months'   gold: '18 months'
Q: When did Lincoln begin his political career?
   pred: '1832'   gold: '1832'
Q: What did The Legal Tender Act of 1862 establish?
   pred: 'United States Note'   gold: 'the United States Note, the first paper currency in United States history'
Q: Who suggested Lincoln grow a beard?
   pred: 'Grace Bedell'   gold: '11-year-old Grace Bedell'
Q: When did the Gettysburg address argue that America was born?
   pred: '1776'   gold: '1776'


# Task 3: Hybrid retrieval with groundedness verification
## 11. Reciprocal Rank Fusion
The BM25 and dense rankings (top 100 each) are fused with RRF: score(p) = Σ 1 / (60 + rank(p)). RRF uses only ranks, so the incompatible score scales of BM25 and cosine similarity don't matter. Passages ranked well by both retrievers rise to the top.

In [14]:
# Reciprocal Rank Fusion: score(p) = sum over retrievers of 1 / (60 + rank of p)
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, p in enumerate(ranking, start=1):
            scores[p] = scores.get(p, 0) + 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

hybrid_ranked = [rrf([bm25_ranked[j], dense_ranked[j]]) for j in range(len(questions))]

print("BM25  :", recall_at_k(bm25_ranked))
print("Dense :", recall_at_k(dense_ranked))
print("Hybrid:", recall_at_k(hybrid_ranked))

BM25  : {'R@1': 0.478, 'R@3': 0.571, 'R@5': 0.602, 'R@10': 0.63, 'n': 508}
Dense : {'R@1': 0.398, 'R@3': 0.528, 'R@5': 0.589, 'R@10': 0.632, 'n': 508}
Hybrid: {'R@1': 0.474, 'R@3': 0.587, 'R@5': 0.622, 'R@10': 0.661, 'n': 508}


## 12. Hybrid RAG evaluation
Same generator, prompt and settings as Task 2, so only the retrieved passages change. Any difference in EM/F1 is therefore due to retrieval.

In [15]:
# same LLM and prompt as Task 2; only the passages change
hybrid_preds = answer_all(hybrid_ranked)

print("Task 1 (BM25 extractive):", evaluate(preds, all_ids), " span:", evaluate(preds, span_ids))
print("Task 2 (dense + LLM)    :", evaluate(dense_preds, all_ids), " span:", evaluate(dense_preds, span_ids))
print("Task 3 (hybrid + LLM)   :", evaluate(hybrid_preds, all_ids), " span:", evaluate(hybrid_preds, span_ids),
      " yes/no:", evaluate(hybrid_preds, yn_ids))

  0%|          | 0/58 [00:00<?, ?it/s]

Task 1 (BM25 extractive): (0.413, 0.459)  span: (0.098, 0.182)
Task 2 (dense + LLM)    : (0.55, 0.641)  span: (0.295, 0.46)
Task 3 (hybrid + LLM)   : (0.564, 0.657)  span: (0.321, 0.488)  yes/no: (0.866, 0.866)


## 13. Lexical-overlap groundedness score
For each generated answer: overlap = (answer content words found in its top-5 passages) / (answer content words). For yes/no answers, which never appear in the text, we use the share of the question's content words found in the best single passage. Low overlap suggests the answer is not supported by the evidence.

In [16]:
# share of the answer's content words found in its top-5 passages (1.0 = fully supported by text)
def overlap_score(answer, question, passage_ids):
    texts = [passages[p] for p in passage_ids[:5]]
    if is_yes_no(answer):
        # yes/no: check the question's claim against the best single passage
        q = set(tokenize(question))
        return max(len(q & set(tokenize(t))) / len(q) for t in texts) if q else 0.0
    a = set(tokenize(answer))
    if not a:
        return 0.0
    evidence = {w for t in texts for w in tokenize(t)}
    return len(a & evidence) / len(a)

# the spec asks to check Task 2's answers; we also check Task 3's
dense_overlaps  = [overlap_score(dense_preds[j],  questions[j], dense_ranked[j])  for j in all_ids]
hybrid_overlaps = [overlap_score(hybrid_preds[j], questions[j], hybrid_ranked[j]) for j in all_ids]

for j in all_ids[:8]:
    print(f"{hybrid_overlaps[j]:.2f}  pred={hybrid_preds[j]!r:28} gold={answers[j]!r}")

1.00  pred='yes'                        gold='yes'
0.71  pred='yes'                        gold='yes'
0.50  pred='no'                         gold='no'
1.00  pred='18 months'                  gold='18 months'
1.00  pred='1832'                       gold='1832'
1.00  pred='United States Note'         gold='the United States Note, the first paper currency in United States history'
1.00  pred='Grace Bedell'               gold='11-year-old Grace Bedell'
1.00  pred='1776'                       gold='1776'


## 14. Groundedness-check evaluation
An answer is flagged as likely ungrounded if overlap < τ. An answer is considered correct if F1 ≥ 0.5. We report:
- **check accuracy** (verdict agrees with correctness)
- the share of **correct answers with high overlap**
- the share of **incorrect answers with low overlap**
- a baseline that never flags anything

These are reported for τ ∈ {0.25, 0.5, 0.75, 1.0}.

In [17]:
# evaluate the overlap check as a classifier: overlap >= tau means "trust"
def check_report(preds, overlaps, tau):
    correct  = [f1_score(preds[j], answers[j]) >= 0.5 for j in all_ids]
    trusted  = [overlaps[j] >= tau for j in all_ids]
    n_right  = sum(correct)
    n_wrong  = len(correct) - n_right
    agree    = sum(c == t for c, t in zip(correct, trusted))
    right_hi = sum(c and t for c, t in zip(correct, trusted))
    wrong_lo = sum(not c and not t for c, t in zip(correct, trusted))
    return {"tau": tau,
            "check_acc": round(agree / len(correct), 3),
            "correct_high": round(right_hi / n_right, 3),
            "wrong_low": round(wrong_lo / n_wrong, 3),
            "flagged": len(correct) - sum(trusted),
            "always_trust": round(n_right / len(correct), 3)}   # baseline: never flag anything

for name, p, o in [("Task 2", dense_preds, dense_overlaps), ("Task 3", hybrid_preds, hybrid_overlaps)]:
    print(name)
    for tau in (0.25, 0.5, 0.75, 1.0):
        print("  ", check_report(p, o, tau))

Task 2
   {'tau': 0.25, 'check_acc': 0.674, 'correct_high': 0.99, 'wrong_low': 0.101, 'flagged': 39, 'always_trust': 0.645}
   {'tau': 0.5, 'check_acc': 0.678, 'correct_high': 0.961, 'wrong_low': 0.163, 'flagged': 76, 'always_trust': 0.645}
   {'tau': 0.75, 'check_acc': 0.658, 'correct_high': 0.856, 'wrong_low': 0.298, 'flagged': 182, 'always_trust': 0.645}
   {'tau': 1.0, 'check_acc': 0.602, 'correct_high': 0.731, 'wrong_low': 0.368, 'flagged': 279, 'always_trust': 0.645}
Task 3
   {'tau': 0.25, 'check_acc': 0.675, 'correct_high': 0.988, 'wrong_low': 0.067, 'flagged': 28, 'always_trust': 0.66}
   {'tau': 0.5, 'check_acc': 0.68, 'correct_high': 0.975, 'wrong_low': 0.106, 'flagged': 48, 'always_trust': 0.66}
   {'tau': 0.75, 'check_acc': 0.664, 'correct_high': 0.87, 'wrong_low': 0.266, 'flagged': 162, 'always_trust': 0.66}
   {'tau': 1.0, 'check_acc': 0.612, 'correct_high': 0.744, 'wrong_low': 0.356, 'flagged': 266, 'always_trust': 0.66}


# Final results

In [18]:
import pandas as pd

# one row per system: QA quality + retrieval recall
rows = []
for name, ranked, p in [("Task 1: BM25 + extractive", bm25_ranked, preds),
                        ("Task 2: Dense + LLM", dense_ranked, dense_preds),
                        ("Task 3: Hybrid RRF + LLM", hybrid_ranked, hybrid_preds)]:
    r = recall_at_k(ranked)
    em, f1 = evaluate(p, all_ids)
    span_em, span_f1 = evaluate(p, span_ids)
    rows.append({"system": name, "EM": em, "F1": f1, "span EM": span_em, "span F1": span_f1,
                 "R@1": r["R@1"], "R@3": r["R@3"], "R@5": r["R@5"], "R@10": r["R@10"]})
display(pd.DataFrame(rows).set_index("system"))

# groundedness check at tau = 0.5
display(pd.DataFrame([check_report(dense_preds, dense_overlaps, 0.5) | {"answers": "Task 2"},
                      check_report(hybrid_preds, hybrid_overlaps, 0.5) | {"answers": "Task 3"}]).set_index("answers"))

,EM,F1,span EM,span F1,R@1,R@3,R@5,R@10
system,,,,,,,,
Task 1: BM25 + extractive,0.413,0.459,0.098,0.182,0.478,0.571,0.602,0.630
Task 2: Dense + LLM,0.550,0.641,0.295,0.460,0.398,0.528,0.589,0.632
Task 3: Hybrid RRF + LLM,0.564,0.657,0.321,0.488,0.474,0.587,0.622,0.661


,tau,check_acc,correct_high,wrong_low,flagged,always_trust
answers,,,,,,
Task 2,0.5,0.678,0.961,0.163,76,0.645
Task 3,0.5,0.680,0.975,0.106,48,0.660


## Conclusions
1. **Retrieval.** BM25 beats dense retrieval at R@1 (0.478 vs 0.398), because the questions reuse the passages' exact names and rare words. Hybrid RRF is the best retriever from R@3 onward (R@5 0.622, R@10 0.661).
2. **Answering.** Generation is the main gain. EM goes from 0.413 (extractive) to 0.550 (dense + LLM), and span EM from 0.098 to 0.295. Hybrid retrieval adds a smaller, controlled gain (EM 0.564, F1 0.657), because only the passages changed.
3. **Groundedness.** At τ = 0.5, the overlap check keeps 96–98% of correct answers but flags only 11–16% of wrong ones. Its accuracy (0.68) is only 2–3 points above the never-flag baseline. Most LLM errors are *grounded but wrong*: a real entity copied from the passages, just the wrong one (e.g. "Kentucky" for "Hardin County"), which word overlap cannot detect.
4. **Limitations.**
   - EM penalises correct answers in a different form ("Grace Bedell" vs "11-year-old Grace Bedell").
   - recall@k by answer containment is a lower bound.
   - Task 1 always answers "yes" to yes/no questions.

   Possible improvements: stemming and number normalisation, a cross-encoder reranker, and an NLI-based groundedness check.